In [1]:
%%bash
python -m src.train_dann \
  --source-csv data/ddr/DR_grading.csv \
  --source-img-dir data/ddr/DR_grading \
  --source-image-col id_code \
  --source-label-col diagnosis \
  --source-default-ext .jpg \
  --target-train-csv data/aptos/target_train_unlabeled.csv \
  --target-train-img-dir data/aptos/train_images \
  --target-train-image-col id_code \
  --target-test-csv data/aptos/target_test.csv \
  --target-test-img-dir data/aptos/train_images \
  --target-test-image-col id_code \
  --target-test-label-col diagnosis \
  --target-test-default-ext .png \
  --label-scheme three \
  --num-classes 3 \
  --backbone efficientnet_b2 \
  --img-size 260 \
  --epochs 15 \
  --batch-size 16 \
  --lr 5e-5 \
  --feature-lr-mult 0.2 \
  --class-head-lr-mult 1.0 \
  --domain-lr-mult 0.25 \
  --domain-loss-weight 0.5 \
  --warmup-epochs 2 \
  --entropy-weight 0.005 \
  --label-smoothing 0.05 \
  --model-selection target_acc \
  --output-dir outputs/dann_ddr_to_aptos_3class

Using device: mps
Label scheme: three
Classification stages: ['Class 0 (No DR): Patient is healthy. See you in a year for your annual checkup.', 'Class 1 (Mild/Moderate): Early signs of disease detected. Schedule a routine appointment with an eye doctor to monitor.', 'Class 2 (Severe/Proliferative): Vision-threatening disease detected. Urgent medical intervention required to prevent blindness.']
Source train class counts: [5013, 4085, 919]
Using class-balanced loss weights: [0.3905, 0.4792, 2.1302]


Epoch 001 | src_train_acc=0.6054 | src_val_f1=0.7126 | target_acc=0.5075 | dom_acc=0.5106


Epoch 002 | src_train_acc=0.7241 | src_val_f1=0.7546 | target_acc=0.5375 | dom_acc=0.5125


Epoch 003 | src_train_acc=0.7621 | src_val_f1=0.7602 | target_acc=0.6385 | dom_acc=0.8664


Epoch 004 | src_train_acc=0.7828 | src_val_f1=0.7847 | target_acc=0.7285 | dom_acc=0.7165


...

Epoch 011 | src_train_acc=0.8372 | src_val_f1=0.8097 | target_acc=0.8186 | dom_acc=0.6231


Epoch 012 | src_train_acc=0.8375 | src_val_f1=0.8116 | target_acc=0.8199 | dom_acc=0.6819


Epoch 013 | src_train_acc=0.8429 | src_val_f1=0.8153 | target_acc=0.8281 | dom_acc=0.6823


Epoch 014 | src_train_acc=0.8442 | src_val_f1=0.8111 | target_acc=0.8199 | dom_acc=0.6717


Epoch 015 | src_train_acc=0.8453 | src_val_f1=0.8055 | target_acc=0.8172 | dom_acc=0.6684
Saved: outputs/dann_ddr_to_aptos_3class/best_dann.pt
Saved: outputs/dann_ddr_to_aptos_3class/metrics.json
Saved: outputs/dann_ddr_to_aptos_3class/target_classification_report.json
Saved: outputs/dann_ddr_to_aptos_3class/target_classification_report.txt


In [2]:
import json
from pathlib import Path

# Pointing exactly to your new 3-class output directory
metrics_path = Path("outputs/dann_ddr_to_aptos_3class/metrics.json")
report_path = Path("outputs/dann_ddr_to_aptos_3class/target_classification_report.txt")

print("Metrics file exists:", metrics_path.exists())
print("Report file exists:", report_path.exists())

if metrics_path.exists():
    m = json.loads(metrics_path.read_text())
    print("Final target:", m.get("final_target"))

if report_path.exists():
    print("\n=== 3-Class Classification Report ===")
    print(report_path.read_text())

Metrics file exists: True
Report file exists: True
Final target: {'loss': 0.4813044111855521, 'accuracy': 0.8281036834924966, 'macro_f1': 0.7440909019046914}

=== 3-Class Classification Report ===
                                                                                                                                 precision    recall  f1-score   support

                                                Class 0 (No DR): Patient is healthy. See you in a year for your annual checkup.     0.9268    0.9474    0.9370       361
        Class 1 (Mild/Moderate): Early signs of disease detected. Schedule a routine appointment with an eye doctor to monitor.     0.7525    0.8102    0.7803       274
Class 2 (Severe/Proliferative): Vision-threatening disease detected. Urgent medical intervention required to prevent blindness.     0.6232    0.4388    0.5150        98

                                                                                                                       accura

In [11]:
import sys
import os
import importlib
import inspect
import torch
import pandas as pd
from PIL import Image
from torchvision import transforms
from sklearn.metrics import classification_report, accuracy_score
from pathlib import Path

# --- 1. The Path Fix (Force Jupyter to the Project Root) ---
PROJECT_ROOT = "/Users/aditya/Desktop/New project"
os.chdir(PROJECT_ROOT)
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

print(f"Current working directory set to: {os.getcwd()}")

# --- 2. Configuration ---
IDRID_CSV_PATH = "data/idrid/idrid_labels.csv" 
IDRID_IMG_DIR = "data/idrid/imagenes"
MODEL_WEIGHTS = "outputs/dann_ddr_to_aptos_3class/best_dann.pt"

DEVICE = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# --- 3. Smart Model Importer ---
print("Hunting for the DANN architecture inside your 'src' folder...")
DANN = None
for filename in os.listdir("src"):
    if filename.endswith(".py") and not filename.startswith("__"):
        mod_name = f"src.{filename[:-3]}"
        try:
            mod = importlib.import_module(mod_name)
            for name, obj in inspect.getmembers(mod, inspect.isclass):
                if name == "DANN":
                    DANN = obj
                    print(f"✅ Successfully found and imported DANN from '{mod_name}'!")
                    break
        except Exception:
            pass
    if DANN:
        break

if DANN is None:
    raise ImportError("Could not find the DANN class in any file inside your 'src' folder!")

# --- 4. Load and Prepare IDRiD Data ---
def map_to_3_class(label):
    if label == 0:
        return 0 # No DR
    elif label in [1, 2]:
        return 1 # Referable (Mild/Moderate)
    elif label in [3, 4]:
        return 2 # Urgent (Severe/Proliferative)
    return label

df = pd.read_csv(IDRID_CSV_PATH)
IMAGE_COL = "Image name" if "Image name" in df.columns else "id_code"
LABEL_COL = "Retinopathy grade" if "Retinopathy grade" in df.columns else "diagnosis"
df['true_3class'] = df[LABEL_COL].apply(map_to_3_class)

test_transforms = transforms.Compose([
    transforms.Resize((260, 260)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# --- 5. Initialize the Model ---
print("Loading model weights...")
model = DANN(backbone='efficientnet_b2', num_classes=3).to(DEVICE)
model.load_state_dict(torch.load(MODEL_WEIGHTS, map_location=DEVICE))
model.eval()

# --- 6. Run Inference ---
predictions = []
true_labels = df['true_3class'].tolist()

print(f"Testing on {len(df)} IDRiD images. This should take less than a minute...")
with torch.no_grad():
    for idx, row in df.iterrows():
        img_name = str(row[IMAGE_COL])
        
        # Try .jpg, then .png, then no extension
        img_path = Path(IDRID_IMG_DIR) / f"{img_name}.jpg" 
        if not img_path.exists():
            img_path = Path(IDRID_IMG_DIR) / f"{img_name}.png"
        if not img_path.exists():
            img_path = Path(IDRID_IMG_DIR) / img_name
            
        img = Image.open(img_path).convert('RGB')
        img_tensor = test_transforms(img).unsqueeze(0).to(DEVICE)
        
        # --- THE FIX: Forward pass without alpha, safe tuple unpacking ---
        outputs = model(img_tensor)
        
        if isinstance(outputs, tuple):
            class_logits = outputs[0]
        else:
            class_logits = outputs
            
        pred_class = torch.argmax(class_logits, dim=1).item()
        predictions.append(pred_class)

# --- 7. Print Results ---
acc = accuracy_score(true_labels, predictions)
print(f"\n=========================================")
print(f"🏆 IDRiD OOD Test Accuracy: {acc:.4f}")
print(f"=========================================\n")

target_names = ['0: No DR', '1: Referable (Mild/Mod)', '2: Urgent (Sev/Prolif)']
print("=== IDRiD Clinical Classification Report ===")
print(classification_report(true_labels, predictions, target_names=target_names))

Current working directory set to: /Users/aditya/Desktop/New project
Using device: mps
Hunting for the DANN architecture inside your 'src' folder...
✅ Successfully found and imported DANN from 'src.models'!
Loading model weights...
Testing on 455 IDRiD images. This should take less than a minute...

🏆 IDRiD OOD Test Accuracy: 0.7187

=== IDRiD Clinical Classification Report ===
                         precision    recall  f1-score   support

               0: No DR       0.84      0.78      0.81       129
1: Referable (Mild/Mod)       0.67      0.62      0.64       178
 2: Urgent (Sev/Prolif)       0.68      0.79      0.73       148

               accuracy                           0.72       455
              macro avg       0.73      0.73      0.73       455
           weighted avg       0.72      0.72      0.72       455

